# TP53 Mutation Analysis — full pipeline on Kaggle
Reproducible ML + structural analysis of disease-associated TP53 variants. **Setup:** turn **Internet ON** (right-side Settings) for the repo clone and live ClinVar fetch. No GPU needed. Runtime ~6–9 min. Every result below is computed live in this notebook.

## 1 · Code and dependencies

In [ ]:
!git clone https://github.com/simbisaichinhema/TP53-Sequence-Structure-and-Machine-Learning-.git 2>&1 | tail -1
%cd TP53-Sequence-Structure-and-Machine-Learning-
!pip install -q pandas scikit-learn matplotlib
import sklearn, pandas, matplotlib
print('sklearn', sklearn.__version__, '| pandas', pandas.__version__, '| matplotlib', matplotlib.__version__)

## 2 · WT baseline (verified reference — reused, not rebuilt)

In [ ]:
!python3 scripts/sequence/verify_sequence.py --reference data/raw/tp53_cds/P04637_uniprot_live_20260918.fasta 2>&1 | tail -6
!python3 scripts/structure/superpose_core.py 2>&1 | tail -4

## 3 · Variant dataset — live ClinVar fetch

In [ ]:
import os
if os.path.exists('data/processed/mutations/tp53_variant_dataset.csv'):
    print('Dataset cached — skipping download. Regenerable via scripts/mutation/fetch_clinvar_tp53.py')
else:
    !python3 scripts/mutation/fetch_clinvar_tp53.py

## 4 · Dataset summary (real counts)

In [ ]:
import pandas as pd
df = pd.read_csv('data/processed/mutations/tp53_variant_dataset.csv')
print(f"n={len(df)} | pathogenic={int((df.label==1).sum())} | benign={int((df.label==0).sum())} | "
      f"dupes={df.duplicated(['pos','ref','alt']).sum()} | missing={df.isna().sum().sum()} | "
      f"positions {df.pos.min()}–{df.pos.max()}")
df[df.hgvs_p.isin(['p.R175H','p.R248Q','p.R248W','p.R273H','p.G245S','p.Y220C','p.P72R'])][['hgvs_p','classification','review_status']]

## 5 · Feature extraction — 21 leak-free sequence/structural features

In [ ]:
!python3 scripts/machine_learning/make_features.py
feat = pd.read_csv('data/processed/structural_features/tp53_features.csv')
print(feat.shape)
feat[['hgvs_p','pos','ref','alt','label','grantham','plddt_at_pos','in_core']].head(5)

## 6 · Train and evaluate (stratified 80/20, seed 42)

In [ ]:
!python3 scripts/machine_learning/train.py

## 7 · Metrics + evaluation figures

In [ ]:
import json
m = json.load(open('results/machine_learning/metrics.json'))
rows = [{'model': n, 'CV-F1': f"{v['cv_f1_mean']}±{v['cv_f1_std']}", 'acc': v['accuracy'],
         'prec': v['precision'], 'rec': v['recall'], 'F1': v['f1'], 'ROC-AUC': v['roc_auc']}
        for n, v in m['models'].items()]
display(pd.DataFrame(rows).set_index('model'))
print('selected:', m['selected_model'], '| top features:', ', '.join(m['feature_importance_ranking'][:5]))
from IPython.display import Image
for f in ['figures/machine_learning/confusion_matrix.png', 'figures/machine_learning/roc_curve.png',
          'figures/machine_learning/feature_importance.png', 'figures/machine_learning/class_distribution.png']:
    display(Image(f, width=620))

## 8 · Real mutant proteins (WT-validated single substitutions)

In [ ]:
!python3 scripts/mutation/build_mutant_sequences.py
mut = pd.read_csv('data/processed/mutations/mutant_sequences.csv')
print('lengths unique:', mut.length.unique(), '| sequences unique:', mut.md5.nunique() == len(mut))
!python3 scripts/mutation/write_variant_protein_results.py

## 9 · WT vs mutant sequence windows (±7 aa, demo hotspots)

In [ ]:
wt = ''.join(l.strip() for l in open('data/processed/protein/WT_TP53_ORF1_393aa.fasta') if l.strip() and not l.startswith('>'))
seqs, cur = {}, None
for line in open('data/processed/mutations/mutant_sequences.fasta'):
    line = line.strip()
    if line.startswith('>'): cur = line.split('|')[1]; seqs[cur] = ''
    else: seqs[cur] += line
for h, pos in [('p.R175H',175),('p.R248Q',248),('p.R273H',273),('p.Y220C',220),('p.P72R',72)]:
    lo, hi = pos-8, pos+7
    print(f"{h}\n  WT    {wt[lo-1:hi]}\n  mutant {seqs[h][lo-1:hi]}\n")

## 10 · Full-cohort audit — all 388 predicted, mistakes listed

In [ ]:
!python3 scripts/machine_learning/audit_predictions.py
pred = pd.read_csv('results/machine_learning/all_predictions.csv')
mis = pred[~pred.correct].sort_values('proba_pathogenic')
print(f"misclassified: {len(mis)} of {len(pred)} (test acc {(pred[pred.split=='test'].correct.mean()):.4f})")
display(mis[['hgvs_p','classification','split','proba_pathogenic']].reset_index(drop=True))
from IPython.display import Image
for f in ['figures/machine_learning/position_enrichment.png', 'figures/machine_learning/plddt_by_class.png',
          'figures/machine_learning/substitution_counts.png']:
    display(Image(f, width=660))

## 11 · Prediction panel — hotspot results from the trained model

In [ ]:
import sys; sys.path.insert(0, 'scripts/machine_learning')
from predict import predict_variant
rows = []
for v in [(175,'R','H'),(248,'R','Q'),(248,'R','W'),(273,'R','H'),(220,'Y','C'),(245,'G','S'),(72,'P','R')]:
    r = predict_variant(*v); c = r['context']
    rows.append({'variant': r['input']['hgvs_p'], 'prediction': r['result'],
                 'P(pathogenic)': r['probability_pathogenic'], 'domain': c['domain'],
                 'pLDDT': c['plddt_at_pos'], 'Grantham': c['grantham']})
display(pd.DataFrame(rows).set_index('variant'))

## 12 · WT structure with hotspot positions (static 3D, real model_4 coordinates)

In [ ]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
tr = json.load(open('dashboard/ca_trace.json'))['residues']
xs = [r['x'] for r in tr]; ys = [r['y'] for r in tr]; zs = [r['z'] for r in tr]; pl = [r['plddt'] for r in tr]
HOT = {175:'R175H', 248:'R248Q', 273:'R273H', 220:'Y220C', 72:'P72R'}
fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection='3d')
ax.plot(xs, ys, zs, color='#9aa3b2', lw=1, alpha=0.7)
sc = ax.scatter(xs, ys, zs, c=pl, cmap='viridis', s=6, vmin=0, vmax=100)
for pos, lab in HOT.items():
    r = tr[pos-1]
    ax.scatter([r['x']],[r['y']],[r['z']], color='#b4232a', s=60)
    ax.text(r['x'], r['y'], r['z'], lab, color='#b4232a', fontsize=9)
fig.colorbar(sc, ax=ax, shrink=0.6, label='pLDDT')
ax.set_title('WT TP53 CA trace (AlphaFold model_4) — red = demo variant positions')
ax.set_axis_off()
fig.tight_layout(); fig.savefig('figures/structures/kaggle_3d_hotspots.png', dpi=150)
from IPython.display import Image
display(Image('figures/structures/kaggle_3d_hotspots.png', width=680))

## 13 · Done — what this notebook proves
WT verified (100% P04637) · 388 traceable ClinVar variants · 388 real mutant proteins · 21 leak-free features · random forest held-out F1 0.88 / ROC-AUC 0.94 · full-cohort audit with explained errors. The HTML dashboard (`dashboard/`) is a local-web demo of this same model and cannot be served from Kaggle.